# Block 3 — pair features

**Reads:** Block 1 records and Block 2 candidates (through `er_eval` / `b2_prod`).
**Writes:** `workspace/block3/v1/` — `features/<split>/<ROLE>/<COUNTRY>/part-*.parquet` (~75
hand-crafted features per candidate pair, groups G1–G9), `vocab/` (frozen token IDF), `g9/`
(cross-list exclusivity pre-pass), `manifest.json`, and `reports/selected_features.json` — the
feature list Blocks 5 and 6 read.

CPU, high RAM. Resumable: a feature part counts only once its `done-*.json` exists, and a logged
bake-off run is skipped.

## Setup (identical first code cell in every notebook)

`BASE` is the folder that holds `src/` — found by walking up from the working directory, or taken
from `$BER_HOME`. Everything then follows: `DATA = BASE/data` (read-only, supplied by the grader),
`WORKSPACE = BASE/workspace` (everything this pipeline writes). `$BER_WORKSPACE` is set to `WORKSPACE`
**before any module is imported**, because `b2_common` / `b2_prod` compute their paths at import
time. `$HF_HOME` keeps model downloads inside the workspace. Nothing outside `BASE` is read or
written except temporary scratch files.

In [ ]:
# Setup — repository layout, workspace, module path
import os
import sys
import tempfile
from pathlib import Path


def find_base(start=None):
    """The repository folder that holds src/. Override with $BER_HOME."""
    if os.environ.get("BER_HOME"):
        return Path(os.environ["BER_HOME"]).expanduser().resolve()
    here = Path(start or os.getcwd()).resolve()
    for d in (here, *here.parents):
        if (d / "src").is_dir():
            return d
    raise FileNotFoundError(f"no folder containing src/ at or above {here}; set BER_HOME")


BASE = find_base()
SRC, DATA, WORKSPACE = BASE / "src", BASE / "data", BASE / "workspace"
WORKSPACE.mkdir(parents=True, exist_ok=True)
ROOT = str(WORKSPACE)

os.environ["BER_WORKSPACE"] = ROOT                       # b2_common / b2_prod read this at import time
os.environ["HF_HOME"] = str(WORKSPACE / "hf_cache")   # model downloads stay inside the workspace
WORK = tempfile.gettempdir()                          # scratch for large local-first writes
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print("BASE     :", BASE)
print("DATA     :", DATA, "" if DATA.is_dir() else "(MISSING — supplied by the grader)")
print("WORKSPACE:", WORKSPACE)

import gc
import json
import shutil
import time

import pandas as pd
import er_eval as ev
import b2_prod as b2p
import b3_features as b3

ev.set_root(ROOT)

# ---------------- config ----------------
B3_VERSION = "v1"                 # bump when features are ADDED/CHANGED (never overwrite a scored version)
TRAIN_C = ["INDIA", "US"]
TEST_C = ["INDIA", "US", "FRANCE"]
PLAN = {"train": {"roles": b3.FEATURE_ROLES, "countries": TRAIN_C},
        "test": {"roles": ["TEST"], "countries": TEST_C}}
WORKER, N_WORKERS = 0, 1          # extra processes for the same country: shard % N_WORKERS == WORKER
USE_G9 = True                     # cross-list exclusivity group (needs the pre-pass in Stage 4)
TREE_FIT_S1_FRAC = 0.20           # the bake-off trains on a 20% subsample of TREE-FIT, BY S1 (hash)
BAKE_TASK_TYPE = "GPU" if shutil.which("nvidia-smi") else "CPU"
REPORT_DIR = os.path.join(ROOT, "block3", B3_VERSION, "reports")
print("BAKE_TASK_TYPE =", BAKE_TASK_TYPE)

## Stage 1 — probe the Block 2 API
Confirms the merged manifest, `K_FINAL`, and the candidate schema.

In [ ]:
# Stage 1 — probe
m = b2p.load_merged_manifest()
K_FINAL = int(m["k_final"])
print("K_FINAL =", K_FINAL, "| K_GEN =", m.get("k_gen"), "| manifest keys:", list(m.keys()))
print("coverage:", json.dumps(m.get("coverage"), default=str)[:800])
meta0, df0 = next(b2p.iter_candidates("train", roles=PLAN["train"]["roles"][:1], countries=TRAIN_C[:1], k=K_FINAL))
print("shard index ->", b3.shard_index(meta0, -1))
print(df0.dtypes.to_string())
assert {"s1_id", "cand_id", "source", "score", "rank", "role", "country", "split", "label"} <= set(df0.columns)
assert df0["rank"].max() <= K_FINAL
del df0

## Stage 2 — create every output folder

In [ ]:
# Stage 2 — folders
b3.prepare_folders(B3_VERSION, PLAN)

## Stage 3 — IDF (once, ~10-20 min)
Token document frequencies over the **train S2+S3 pool** only. No labels. Frozen and reused for test.

In [ ]:
# Stage 3 — IDF
IDF_META = os.path.join(b3.ver_dir(B3_VERSION), "vocab", "idf_meta.json")
if not os.path.exists(IDF_META):
    b3.fit_idf(B3_VERSION)
print(json.load(open(IDF_META)))
IDF = b3.load_idf(B3_VERSION)

## Stage 4 — smoke check on one shard (writes nothing)
Throughput, NaN rates, value ranges, row-count / uniqueness / label passthrough. The G9 pre-pass for
the first train country runs first because the smoke features need it.

In [ ]:
# Stage 4 — smoke check
c = TRAIN_C[0]
S, P = b3.load_tables("train", c, IDF)
if USE_G9 and not os.path.exists(b3.g9_path(B3_VERSION, "train", c).replace(".parquet", ".done.json")):
    b3.g9_prepass(B3_VERSION, b2p, "train", c, P, K_FINAL)
g9 = b3.load_g9(B3_VERSION, "train", c, P) if USE_G9 else None
meta, df = next(b2p.iter_candidates("train", roles=PLAN["train"]["roles"][-1:], countries=[c], k=K_FINAL))
keep = df["s1_id"].drop_duplicates().iloc[:1000]
df = df[df["s1_id"].isin(keep)]
t0 = time.time()
feats = b3.featurize(df, S, P, IDF, g9)
el = time.time() - t0
print(f"{len(df):,} rows in {el:.1f}s -> {len(df) / el * 60:,.0f} rows/min (target >= 1,000,000)")
assert len(feats) == len(df) and not feats.duplicated(["s1_id", "cand_id"]).any()
chk = feats.merge(df[["s1_id", "cand_id", "label"]], on=["s1_id", "cand_id"], suffixes=("", "_b2"))
assert (chk["label"].fillna(-1) == chk["label_b2"].fillna(-1)).all(), "label passthrough broken"
num = [f for f in b3.FEATURE_NAMES if f not in b3.CAT_FEATURES]
desc = feats[num].describe().T[["mean", "min", "max"]]
desc["nan_rate"] = feats[num].isna().mean()
desc["pos_mean"] = feats.loc[feats.label == 1, num].mean()
desc["neg_mean"] = feats.loc[feats.label == 0, num].mean()
with pd.option_context("display.max_rows", 200):
    print(desc.round(3).to_string())
del S, P, g9, df, feats, chk, desc
gc.collect()

## Stage 5 — G9 pre-pass and the full feature build
Per (split, country): the record tables are loaded once, the G9 pre-pass counts — for every candidate
record — how many S1 lists (top `K_FINAL`, **all roles of the split**) contain it and its best /
runner-up retrieval score (label-free), then every committed Block 2 shard is featurized. One
(split, country) at a time keeps memory flat.

In [ ]:
# Stage 5 — G9 pre-pass + build
b3.require_folders_ready(B3_VERSION)
for split, p in PLAN.items():
    for c in p["countries"]:
        S, P = b3.load_tables(split, c, IDF)
        if USE_G9 and not os.path.exists(b3.g9_path(B3_VERSION, split, c).replace(".parquet", ".done.json")):
            b3.g9_prepass(B3_VERSION, b2p, split, c, P, K_FINAL)
        summ = b3.run_features(B3_VERSION, b2p, split, p["roles"], [c], K_FINAL, worker=WORKER,
                               n_workers=N_WORKERS, use_g9=USE_G9, tables=(S, P))
        if len(summ):
            print(summ.groupby(["role", "country"])[["n_rows", "elapsed_s"]].sum())
        del S, P, summ
        gc.collect()

## Stage 6 — manifest

In [ ]:
# Stage 6 — manifest
man = b3.write_manifest(B3_VERSION, PLAN)
print(json.dumps(man["parts"], indent=1))

## Stage 7 — leakage scan and redundancy
TREE-FIT (20% by S1) + TREE-DEV. Leakage rule: any single feature with ROC-AUC >= 0.99 is
investigated **before** any bake-off run.

In [ ]:
# Stage 7 — load + leakage scan
tr = b3.load_features(B3_VERSION, "train", ["TREE-FIT"], TRAIN_C, s1_frac=TREE_FIT_S1_FRAC)
dv = b3.load_features(B3_VERSION, "train", ["TREE-DEV"], TRAIN_C)
print(f"TREE-FIT sub: {len(tr):,} rows / {tr.s1_id.nunique():,} S1 | TREE-DEV: {len(dv):,} rows / {dv.s1_id.nunique():,} S1")
UNIVERSE = ev.load_eval_universe(["TREE-DEV"], TRAIN_C)
SCORER = ev.S1Scorer(UNIVERSE)
print("TREE-DEV universe:", len(UNIVERSE), "S1 | singletons", int(UNIVERSE.singleton.sum()),
      "| cross-script S1", int(UNIVERSE.has_cross.sum()))
ALL_FEATS = list(b3.FEATURE_NAMES) if USE_G9 else [f for f in b3.FEATURE_NAMES if f not in b3.GROUP_FEATURES["G9"]]
scan = b3.single_feature_scan(tr, ALL_FEATS)
print(scan.head(25).to_string())
if scan.LEAK_SUSPECT.any():
    print("!!! LEAK SUSPECTS - investigate before continuing:", scan[scan.LEAK_SUSPECT].feature.tolist())
red = b3.redundant_pairs(tr, ALL_FEATS, thr=0.97)
print("near-duplicate feature pairs (|spearman| >= 0.97):")
print(red.head(30).to_string())

## Stage 8 — group bake-off
Fixed CatBoost (depth 6, lr 0.1, <= 1000 iterations, early stop on TREE-DEV). Every run is logged to
`reports/runs.jsonl` with its per-S1 F0.5 vector, so any two runs can be compared with a paired
bootstrap. For a leave-one-group-out variant `delta_vs_ref` is measured against `full`, so a
**negative** delta means the group helps; `WITHIN NOISE` means it is not earning its place.

In [ ]:
# Stage 8 — bake-off
CAT = [c for c in b3.CAT_FEATURES if c in ALL_FEATS]
VARIANTS = {"full": ALL_FEATS, "G1_only": b3.GROUP_FEATURES["G1"]}
for g in b3.GROUPS:
    if g == "G1" or (g == "G9" and not USE_G9):
        continue
    VARIANTS[f"minus_{g}"] = [f for f in ALL_FEATS if f not in b3.GROUP_FEATURES[g]]
VARIANTS["no_country"] = [f for f in ALL_FEATS if f != "country"]
RESULTS = {}
done_ids = set(ev.load_runs(REPORT_DIR).get("run_id", pd.Series(dtype=str)))
for name, feats_ in VARIANTS.items():
    rid = f"{B3_VERSION}_{name}"
    if rid in done_ids:
        print("skip (logged):", rid)
        continue
    print(f"\n=== {rid}: {len(feats_)} features")
    model, prob, info = ev.proxy_run(rid, tr, dv, feats_, CAT, SCORER, REPORT_DIR, variant=name,
                                     ref_run=None if name == "full" else f"{B3_VERSION}_full",
                                     task_type=BAKE_TASK_TYPE,
                                     extra={"b3_version": B3_VERSION, "feature_set_hash": b3.feature_set_hash()})
    RESULTS[name] = (model, prob, info)
    print(f"  S1-macro F0.5 {info['s1_macro_f05']:.4f} [{info['ci95_low']:.4f}, {info['ci95_high']:.4f}]  "
          f"PR-AUC {info['pr_auc']:.4f}  thr {info['threshold']:.3f}  "
          + (f"delta vs full {info['delta_vs_ref']['delta']:+.4f} ({info['delta_vs_ref']['verdict']})" if "delta_vs_ref" in info else ""))
print(ev.summary_table(REPORT_DIR).to_string())

## Stage 9 — the full run in detail

In [ ]:
# Stage 9 — slices and importance of the full run
runs = ev.load_runs(REPORT_DIR)
full = runs[runs.run_id == f"{B3_VERSION}_full"].iloc[-1]
print("FULL:", {k: round(v, 4) for k, v in full.items() if k.startswith("slices_s1.") and k.endswith("s1_macro_f05")})
print("pair PR-AUC by slice:", {k: round(v, 4) for k, v in full.items() if k.startswith("slices_pair.") and k.endswith("pr_auc")})
print("threshold", round(full.threshold, 4), "| accepted/S1", round(full.accepted_per_s1_mean, 3),
      "| singletons with a false accept", full.singletons_with_false_accept,
      "| retrieval recall ceiling", round(full.retrieval_recall_ceiling, 4))
print("top-15 importance:", full.importance_top15)

## Stage 10 — the selected feature set + error analysis
Decision rule: S1-macro F0.5 first, then PR-AUC; drop a group only if `minus_Gx` is `WITHIN NOISE` or
`BETTER`, and prefer the smaller set within noise. Both drop lists were empty in the submitted run, so
`selected_features.json` holds the full list. Dropping features later never requires recomputation;
**adding** one means a new `B3_VERSION` and rebuilding every role.

In [ ]:
# Stage 10 — select + error analysis
DROP_GROUPS   = []        # e.g. ["G4"] if minus_G4 was WITHIN NOISE
DROP_FEATURES = []        # e.g. ["country"] if no_country was WITHIN NOISE / BETTER (safer for FRANCE)
SELECTED = [f for f in ALL_FEATS if f not in DROP_FEATURES and not any(f in b3.GROUP_FEATURES[g] for g in DROP_GROUPS)]
if SELECTED != ALL_FEATS:
    rid = f"{B3_VERSION}_selected_{len(SELECTED)}"
    model, prob, info = ev.proxy_run(rid, tr, dv, SELECTED, [c for c in CAT if c in SELECTED], SCORER, REPORT_DIR,
                                     variant="selected", ref_run=f"{B3_VERSION}_full", task_type=BAKE_TASK_TYPE)
    print(rid, round(info["s1_macro_f05"], 4), info.get("delta_vs_ref"))
else:
    model, prob, info = RESULTS["full"] if "full" in RESULTS else ev.proxy_run(
        f"{B3_VERSION}_full_rerun", tr, dv, ALL_FEATS, CAT, SCORER, REPORT_DIR, variant="full", task_type=BAKE_TASK_TYPE)
json.dump({"b3_version": B3_VERSION, "features": SELECTED, "cat_features": [c for c in CAT if c in SELECTED],
           "feature_set_hash": b3.feature_set_hash(SELECTED), "time": time.strftime("%F %T")},
          open(os.path.join(REPORT_DIR, "selected_features.json"), "w"), indent=1)
fp, fn = b3.error_examples(dv, prob, info["threshold"], n=50)
fp.to_csv(os.path.join(REPORT_DIR, "top_false_positives.csv"), index=False)
fn.to_csv(os.path.join(REPORT_DIR, "top_false_negatives.csv"), index=False)
cols = ["prob", "country", "dense_rank", "s1_name", "cand_name", "s1_addr", "cand_addr"]
print("TOP FALSE POSITIVES\n" + fp[cols].head(20).to_string())
print("TOP FALSE NEGATIVES\n" + fn[cols].head(20).to_string())
print("\nBlock 3 done. Next: 04_block4.ipynb")